# Generowanie tekstu w stylu Szekspira przy użyciu sieci rekurencyjnych (GRU)
Michał Wilk, Wiktor Zmiendak

## Cel ćwiczenia
Celem zadania było stworzenie, wytrenowanie i przetestowanie modelu rekurencyjnej sieci neuronowej (RNN) z wykorzystaniem warstw GRU (Gated Recurrent Unit), zdolnego do generowania tekstu naśladującego styl Williama Szekspira (predykcja znak po znaku).

## Przygotowanie i przetwarzanie danych
Jako zbiór uczący wykorzystaliśmy wszystkie dzieła Szekspira.

In [1]:
import tensorflow as tf
shakespeare_url = "https://homl.info/shakespeare" # shortcut URL
filepath = tf.keras.utils.get_file("shakespeare.txt", shakespeare_url)
with open(filepath) as f:
  shakespeare_text = f.read()

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Wyświetlamy kilka pierwszych wierszy

In [2]:
print(shakespeare_text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


### Wektoryzacja tekstu
Surowy tekst poddaliśmy wstępnemu przetwarzaniu:

 - Tokenizacji: Podział na znaki (char-level).

 - Standaryzacji: Konwersja na małe litery.

 - Kodowaniu: Każdy unikalny znak został przypisany do liczby całkowitej.

Odrzuciliśmy tokeny systemowe (0 - padding, 1 - unknown), przesuwając indeksy o 2, aby operować na czystych identyfikatorach znaków.

In [3]:
text_vec_layer = tf.keras.layers.TextVectorization(
  split="character",
  standardize="lower"
)
text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0]
encoded -= 2 # drop tokens 0 (pad) and 1 (unknown), which we will not use
n_tokens = text_vec_layer.vocabulary_size() - 2 # number of distinct chars = 39
dataset_size = len(encoded) # total number of chars = 1,115,394

### Tworzenie sekwencji (Dataset)

Dane przekształciliśmy w sekwencje uczące za pomocą techniki **okienkowania (windowing)**:

- **Wejście ($X$):** Sekwencja o długości $L$.
    
- **Wyjście ($Y$):** Ta sama sekwencja przesunięta o jeden "takt" (krok czasowy) w przyszłość.
    
- **Długość ramki (length):** Ustalona na 100 znaków.
    
- zastosowaliśmy tasowanie (`shuffle`), grupowanie (`batch_size=32`) oraz wstępne ładowanie (`prefetch`) dla optymalizacji wydajności.
    

Zbiór danych podzieliliśmy następująco:

- Zbiór treningowy: 90% (pierwszy 1 mln znaków).
    
- Zbiór walidacyjny: 5%.
    
- Zbiór testowy: 5%.

In [4]:
def to_dataset(sequence, length, shuffle=False, seed=None, batch_size=32):
  ds = tf.data.Dataset.from_tensor_slices(sequence)
  ds = ds.window(length + 1, shift=1, drop_remainder=True)
  ds = ds.flat_map(lambda window_ds: window_ds.batch(length + 1))
  if shuffle:
    ds = ds.shuffle(100_000, seed=seed)

  ds = ds.batch(batch_size)
  return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

In [5]:
list(to_dataset(text_vec_layer(["To be"])[0], length=4))

[(<tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[ 4,  5,  2, 23]], dtype=int64)>,
  <tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[ 5,  2, 23,  3]], dtype=int64)>)]

In [6]:
length = 100
tf.random.set_seed(42)
train_set = to_dataset(encoded[:1_000_000], length=length, shuffle=True,
seed=42)
valid_set = to_dataset(encoded[1_000_000:1_060_000], length=length)
test_set = to_dataset(encoded[1_060_000:], length=length)

## Architektura Modelu

Ze względu na rozmiar zbioru danych zastosowaliśmy architekturę zdolną do zapamiętywania dłuższych zależności w tekście.

Model składa się z następujących warstw:

1. **Embedding Layer:**
    
    - Wejście: Identyfikatory znaków.
        
    - Wyjście: Wektor o wymiarze 16.
        
    - Cel: Mapowanie dyskretnych znaków na gęstą reprezentację wektorową.
        
    - Wymiarowość wyjścia warstwy: _[batch\_size, window\_length, embedding\_dim]_.
        
2. **GRU Layer (Gated Recurrent Unit):**
    
    - Liczba jednostek: 128.
        
    - Parametr `return_sequences=True`: Warstwa zwraca wynik dla każdego kroku czasowego (nie tylko dla ostatniego), co jest kluczowe przy predykcji "znak po znaku" w całej sekwencji.
        
3. **Dense Layer (Warstwa wyjściowa):**
    
    - Liczba neuronów: 39 (odpowiada liczbie unikalnych znaków).
        
    - Funkcja aktywacji: **Softmax**.
        
    - Cel: Zwrócenie rozkładu prawdopodobieństwa dla każdego z 39 możliwych kolejnych znaków.
        

## Proces Uczenia

Model skompilowaliśmy i trenowaliśmy z następującymi hiperparametrami:

- **Funkcja straty:** `sparse_categorical_crossentropy` (odpowiednia dla klasyfikacji wieloklasowej, gdzie etykiety są liczbami całkowitymi).
    
- **Optymalizator:** `nadam`.
    
- **Metryka:** `accuracy` (dokładność).
    
- **Liczba epok:** 10.
    
- **Callback:** `ModelCheckpoint` – zapisywanie tylko najlepszego modelu na podstawie `val_accuracy`.

In [7]:
tf.random.set_seed(42) # extra code ensures reproducibility on CPU
model = tf.keras.Sequential([
  tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
  tf.keras.layers.GRU(128, return_sequences=True),
  tf.keras.layers.Dense(n_tokens, activation="softmax")
])
model.compile(
  loss="sparse_categorical_crossentropy",
  optimizer="nadam",
  metrics=["accuracy"]
)
model_ckpt = tf.keras.callbacks.ModelCheckpoint(
  "my_shakespeare_model.keras",
  monitor="val_accuracy",
  save_best_only=True
)
history = model.fit(
  train_set,
  validation_data=valid_set,
  epochs=10,
  callbacks=[model_ckpt]
)

Epoch 1/10
   1070/Unknown 110s 85ms/step - accuracy: 0.3168 - loss: 2.4167

KeyboardInterrupt: 

## Implementacja modelu końcowego (Inference)

Po wytrenowaniu, surowy model (operujący na liczbach) został opakowany w strukturę "end-to-end", aby umożliwić przyjmowanie tekstowych danych wejściowych:

1. **Warstwa wejściowa:** `TextVectorization` (zamiana tekstu na indeksy).
    
2. **Warstwa Lambda:** Korekta indeksów (odejmowanie 2).
    
3. **Model właściwy:** Wytrenowany model GRU.

In [8]:
shakespeare_model = tf.keras.Sequential([
  text_vec_layer,
  tf.keras.layers.Lambda(lambda X: X - 2), # no <PAD> or <UNK> tokens
  model
])

### Przykłady predykcji

Dla wejścia _"To be or not to b"_:

1. Model zwraca prawdopodobieństwa dla ostatniego znaku.
    
2. Wyznaczany jest indeks o najwyższym prawdopodobieństwie (`argmax`).
    
3. Indeks zamieniany jest z powrotem na znak w tym przypadku "e".

In [9]:
y_proba = shakespeare_model.predict(tf.constant(["To be or not to b"], dtype=tf.string))[0, -1]
y_pred = tf.argmax(y_proba) # choose the most probable character ID
text_vec_layer.get_vocabulary()[y_pred + 2]


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 773ms/step


'e'

Widzimy też że predykcja działa poprawnie gdy początek sekwencji jest skrócony.

In [11]:
y_proba = shakespeare_model.predict(tf.constant(["not to b"], dtype=tf.string))[0, -1]
y_pred = tf.argmax(y_proba) # choose the most probable character ID
text_vec_layer.get_vocabulary()[y_pred + 2]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step


'e'

Natomiast dla jednego z fragmentów w którym prawidłową predykcją powinno być `k` otrzymujemy `r` ze względu na wyższe prawdopodobieństwo wystąpienia tej litery, zapewne przez nazwisko autora utworów na których model był trenowany.

In [17]:
y_proba = shakespeare_model.predict(tf.constant(["Speak ,spea"], dtype=tf.string))[0, -1]
y_pred = tf.argmax(y_proba) # choose the most probable character ID
text_vec_layer.get_vocabulary()[y_pred + 2]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step


'k'

### Sampling zamiast dekodowania zachłannego (greedy)
W poprzednich komórkach pokazano, jak model może przewidywać najprawdopodobniejszy znak.
Dekodowanie zachłanne zawsze wybiera najbardziej prawdopodobny znak i szybko prowadzi do powtarzalnych, mało ciekawych tekstów.

Zamiast tego stosujemy **próbkowanie** z rozkładu wyjściowego modelu:
 - Najpierw przekształcamy prawdopodobieństwa w logity i dzielimy przez *temperaturę*.
 - Następnie losujemy indeks klasy za pomocą `tf.random.categorical` z tych przeskalowanych logitów.

Efekt temperatury:
 - `temperature -> 0` : zachłanne zachowanie (mniej różnorodności).
 - `temperature = 1` : domyślne losowanie zgodne z przewidzianymi prawdopodobieństwami.
 - `temperature > 1` : wyrównanie rozkładu — więcej kreatywności, ale też więcej błędów.

W komórkach poniżej znajdują się implementacje `next_char` i `extend_text`, które wykonują próbkowanie z temperaturą oraz przykłady generacji.
Dodatkowe opcje, które możesz rozważyć: `top-k` sampling (ograniczenie wyboru do k najbardziej prawdopodobnych znaków) oraz `top-p` (nucleus sampling) — oba zwykle poprawiają jakość generowanego tekstu w zadaniach twórczych.

In [18]:
# Sampling-based next_char and extend_text using temperature
def next_char(text, temperature=1.0):
    # Predict distribution for next character (model expects batch of strings)
    y_proba = shakespeare_model.predict(tf.constant([text], dtype=tf.string))[0, -1]
    # convert to tensor and ensure batch dim for tf.random.categorical
    y_proba = tf.convert_to_tensor(y_proba[None, :], dtype=tf.float32)
    # numerical stability and rescale by temperature
    rescaled_logits = tf.math.log(tf.clip_by_value(y_proba, 1e-9, 1.0)) / float(temperature)
    char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0].numpy()
    # map back to vocabulary (+2 because we dropped PAD/UNK earlier)
    return text_vec_layer.get_vocabulary()[char_id + 2]

def extend_text(text, n_chars=50, temperature=1.0):
    out = text
    for _ in range(n_chars):
        c = next_char(out, temperature=temperature)
        out += c
    return out

In [ ]:
# Examples: generate with different temperatures
tf.random.set_seed(42)  # reproducible sampling on CPU
print(extend_text("To be or not to be", n_chars=100, temperature=0.01))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━

In [22]:
print(extend_text("To be or not to be", n_chars=100, temperature=0.05))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━

In [23]:
print(extend_text("To be or not to be", n_chars=100, temperature=0.10))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━

In [24]:
print(extend_text("To be or not to be", n_chars=100, temperature=1.0))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━

In [ ]:
print(extend_text("To be or not to be", n_chars=100, temperature=5.0))

In [ ]:
print(extend_text("To be or not to be", n_chars=100, temperature=10.0))